In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
import torchvision.models as models
from torchvision import transforms
import numpy as np
from tqdm import tqdm
from utils.dataloader import *


from dotenv import load_dotenv
import os
from Model.models import SimCLRModel

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def plot_pca_scatterplot(X, Y, output_path="pca_scatterplot.jpg", 
                         title="PCA Scatterplot (2 Main Components)",
                         figsize=(12, 8), dpi=150):
    """
    Create a scatterplot of 2 main principal components with different colors/symbols per class.
    
    Args:
        X (array-like): Feature matrix of shape (n_samples, n_features)
        Y (array-like): Labels of shape (n_samples,)
        output_path (str): Path to save the output image
        title (str): Title of the plot
        figsize (tuple): Figure size (width, height)
        dpi (int): DPI for saved image
    
    Returns:
        PCA: Fitted PCA object
    """
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", 
                exist_ok=True)
    
    # Fit PCA with 2 components
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    # Get unique classes and colors
    unique_classes = np.unique(Y)
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_classes)))
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', 'H', '+', 'x', 
               'c', 'd', '|', '_', 'P', 'X', '.', ',']
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each class
    for idx, class_label in enumerate(unique_classes):
        mask = Y == class_label
        marker = markers[idx % len(markers)]
        color = colors[idx]
        
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                  label=f"Class {class_label}",
                  marker=marker,
                  color=color,
                  s=100,
                  alpha=0.7,
                  edgecolors='black',
                  linewidth=0.5)
    
    # Labels and title
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)", fontsize=12)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Grid and legend
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10, framealpha=0.9)
    
    # Add variance info
    total_variance = pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]
    ax.text(0.02, 0.98, f"Total variance explained: {total_variance:.2%}", 
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    # Save figure
    plt.savefig(output_path, format='jpg', dpi=dpi, bbox_inches='tight')
    print(f"✓ Scatterplot saved to: {output_path}")
    print(f"  PC1 explains: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  PC2 explains: {pca.explained_variance_ratio_[1]:.2%}")
    
    plt.show()
    
    return pca